# Spot distribution classification

Train and inspect an InceptionV3 classifier for Clustered, Scattered, Speckled, and Varied patterns. This notebook retains three original training variants; select one rather than running all variants.

**Research code with separate pretrained artifacts.** Extract the model Release asset at the repository root. No images, labels, feature tables, or saved outputs are included. Read [the execution guide](../docs/EXECUTION.md) and [review notes](../docs/REVIEW_NOTES.md) before running experiments.

Paper: https://doi.org/10.1007/s44196-026-01458-x

## Path configuration
Set the project root below. Dataset requests: naoum1496@gmai.com.

In [ ]:
from pathlib import Path
import os

# Run from the repository root or its notebooks/ directory.
# For another location, set STRAWBERRY_PROJECT_ROOT or edit PROJECT_ROOT.
working_dir = Path.cwd()
default_root = working_dir.parent if working_dir.name == "notebooks" else working_dir
PROJECT_ROOT = Path(os.environ.get("STRAWBERRY_PROJECT_ROOT", str(default_root))).expanduser().resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FEATURE_DIR = DATA_DIR / "features"
for directory in (MODEL_DIR, MODEL_DIR / "SVM_PCA_GLCM", RESULTS_DIR, FEATURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)


## 1. Imports and image inventory

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
import shutil

In [ ]:
# Base directory of your dataset
base_dir = f'{DATA_DIR}/Spots_Distribution'

# Step 1: Count the number of images in each folder
for category in ['Clustered', 'Scattered', 'Speckled','Varied']:
    path = os.path.join(base_dir, category)
    num_images = len(os.listdir(path))
    print(f"{category} has {num_images} images.")

In [ ]:
# Step 2: Visualize three images from each class
for category in ['Clustered', 'Scattered', 'Speckled','Varied']:
    path = os.path.join(base_dir, category)
    images = os.listdir(path)[:3]  # Get the first three images
    plt.figure(figsize=(10, 5))
    for i, image_name in enumerate(images):
        img_path = os.path.join(path, image_name)
        img = plt.imread(img_path)
        plt.subplot(1, 3, i + 1)
        plt.imshow(img)
        plt.title(f"{category}")
        plt.axis('off')
    plt.show()

## 2. Image paths and labels

In [ ]:
import os

base_dir = f'{DATA_DIR}/Spots_Distribution'
categories = ['Clustered', 'Scattered', 'Speckled','Varied']
images = []
labels = []

for label, category in enumerate(categories):
    category_dir = os.path.join(base_dir, category)
    for image_name in os.listdir(category_dir):
        images.append(os.path.join(category_dir, image_name))
        labels.append(label)  # Assuming labels are 0 for Early_Stage, 1 for Medium_Stage, and 2 for Severe_Stage


## 3. Shared model and generator definitions

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.applications.inception_v3 import InceptionV3




In [ ]:
def get_data_generator(image_paths, labels, batch_size=32):
    datagen = ImageDataGenerator(rescale=1./255)  # Add any required augmentations here
    return datagen.flow_from_dataframe(
        dataframe=pd.DataFrame({'filename': image_paths, 'class': labels.astype(str)}),
        x_col='filename',
        y_col='class',
        target_size=(224, 224),  # Assuming InceptionV3 input size
        batch_size=batch_size,
        class_mode='categorical')


In [ ]:
# Define a function to create your model
def create_model(input_shape=(224, 224, 3)):
    # Load InceptionV3 as base model
    base_model = InceptionV3(include_top=False, weights='imagenet', input_shape=input_shape)
    # Fine-tuning configuration
    base_model.trainable = True
    fine_tune_at = 100
    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    # Adding custom layers
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(1024, activation='relu')(x)
    predictions = Dense(4, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)

    # Compile the model
    model.compile(optimizer=SGD(learning_rate=1e-5, momentum=0.9),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

## 4. Training variant A

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
import numpy as np
import matplotlib.pyplot as plt

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

labels_np = np.array(labels)

# Stores histories for all folds
all_fold_histories = []

best_models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(images, labels_np)):
    print(f"Training on fold {fold+1}/{n_splits}...")

    # Split data for the current fold
    X_train, X_val = np.array(images)[train_idx], np.array(images)[val_idx]
    y_train, y_val = np.array(labels)[train_idx], np.array(labels)[val_idx]

    # Create a new instance of the model
    model = create_model()

    # Get data generators
    train_generator = get_data_generator(X_train, y_train)
    val_generator = get_data_generator(X_val, y_val)

    # Path to save the best model for this fold
    best_model_path = f'best_model_fold_{fold+1}.h5'

    # Fit the model
    history = model.fit(
        train_generator,
        epochs=350,
        validation_data=val_generator,
        callbacks=[
            ModelCheckpoint(filepath=best_model_path, save_best_only=True, monitor='val_accuracy', mode='max'),
            EarlyStopping(monitor='val_loss', patience=5, verbose=1, mode='min'),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, verbose=1, mode='min')
        ]
    )

    # Save the history from this fold
    all_fold_histories.append(history.history)

    # Evaluate the best model for this fold
    best_model = load_model(best_model_path)
    val_loss, val_accuracy = best_model.evaluate(val_generator)
    print(f"Fold {fold+1} Validation Accuracy: {val_accuracy}")

    best_models.append((best_model, val_accuracy))


## 5. Training-history plots and export

In [ ]:
# Required imports
import numpy as np
import matplotlib.pyplot as plt

# Assuming 'all_fold_histories' contains the training histories for each fold
# and 'best_models' contains the best model from each fold alongside its validation accuracy

# Initialize dictionaries to hold the summed and average metrics for plotting
summed_history = {
    'accuracy': [],
    'val_accuracy': [],
    'loss': [],
    'val_loss': []
}

average_history = {
    'accuracy': [],
    'val_accuracy': [],
    'loss': [],
    'val_loss': []
}

# Prepare the summed_history with zeros for accurate summation
# This step ensures that each metric's list starts with zeros, matching the length of the epochs trained
for key in summed_history:
    summed_history[key] = [0] * len(all_fold_histories[0][key])

# Sum up the metric values across all folds
for fold_history in all_fold_histories:
    for key in summed_history:
        # Sum corresponding metric values across folds
        summed_history[key] = [sum(x) for x in zip(summed_history[key], fold_history[key])]

# Compute the average of each metric across all folds
for key in average_history:
    # Divide each summed metric by the number of folds to get the average
    average_history[key] = [x / len(all_fold_histories) for x in summed_history[key]]

# Plotting
# Determine the number of epochs from the length of any metric's history
epochs = range(1, len(average_history['accuracy']) + 1)

# Set up figure for plotting accuracy and loss
plt.figure(figsize=(14, 5))

# Plot training and validation accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs, average_history['accuracy'], 'bo-', label='Training Accuracy')
plt.plot(epochs, average_history['val_accuracy'], 'r^-', label='Validation Accuracy')
plt.title('Training and Validation Accuracy per Epoch')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot training and validation loss
plt.subplot(1, 2, 2)
plt.plot(epochs, average_history['loss'], 'bo-', label='Training Loss')
plt.plot(epochs, average_history['val_loss'], 'r^-', label='Validation Loss')
plt.title('Training and Validation Loss per Epoch')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Adjust layout to prevent overlap
plt.tight_layout()

# Display the plots
plt.show()

# Selecting the Best Overall Model
# 'best_models' is assumed to contain tuples of (model, validation_accuracy) for each fold
# Select the model with the highest validation accuracy
best_model_overall = max(best_models, key=lambda item: item[1])[0]

# Save the best overall model to disk
best_model_overall.save(f'{MODEL_DIR}/best_model_overall.h5')


In [ ]:
# Selecting the Best Overall Model along with its accuracy
# 'best_models' is assumed to contain tuples of (model, validation_accuracy) for each fold
best_model_overall, best_model_overall_accuracy = max(best_models, key=lambda item: item[1])

# Save the best overall model to disk
#best_model_overall.save('best_model_overall.h5')

# Print the best overall model's validation accuracy
print("Best Overall Model Accuracy:", best_model_overall_accuracy)


## 6. Single-image inference (optional)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Define categories
categories = ['Clustered', 'Scattered', 'Speckled', 'Varied']

# Load the pre-trained model
model = tf.keras.models.load_model(f'{MODEL_DIR}/Spots_Distribution_best_model_overall.h5')

# Load and preprocess the image
img_path = f'{DATA_DIR}/StrawberryLeafScorch(1763).JPG'  # Replace with the path to your image
img_height, img_width = 224, 224  # Example image dimensions
img = image.load_img(img_path, target_size=(img_height, img_width))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension
input_data = img_array

# Make prediction
prediction = model.predict(input_data)

# Convert probabilities to class label
predicted_class_index = np.argmax(prediction)
predicted_category = categories[predicted_class_index]

# Print out the predicted category
print("Predicted category:", predicted_category)
# Display the image
plt.imshow(img)
plt.title('Predicted Category: ' + predicted_category)
plt.axis('off')
plt.show()


## 7. Training variant B: precision and recall

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.models import load_model, save_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator


def get_data_generator(image_paths, labels, batch_size=32, is_validation=False):
    datagen = ImageDataGenerator(rescale=1./255)  # Add any required augmentations here
    shuffle = not is_validation
    return datagen.flow_from_dataframe(
        dataframe=pd.DataFrame({'filename': image_paths, 'class': labels.astype(str)}),
        x_col='filename',
        y_col='class',
        target_size=(224, 224),  # Assuming InceptionV3 input size
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=shuffle)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
labels_np = np.array(labels)

best_model_accuracy = -1
best_model_path = ''
fold_metrics = []
all_fold_histories = []

for fold, (train_idx, val_idx) in enumerate(skf.split(images, labels_np)):
    print(f"Training on fold {fold+1}/{n_splits}...")
    # Split data for the current fold
    X_train, X_val = np.array(images)[train_idx], np.array(images)[val_idx]
    y_train, y_val = np.array(labels)[train_idx], np.array(labels)[val_idx]

    model = create_model()

    # Get data generators
    # For training generator
    train_generator = get_data_generator(X_train, y_train, is_validation=False)

    # For validation generator
    val_generator = get_data_generator(X_val, y_val, is_validation=True)

    callbacks = [
        ModelCheckpoint(f'{MODEL_DIR}/model_{fold+1}.h5', save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(patience=10, monitor='val_loss'),
        ReduceLROnPlateau(patience=5, monitor='val_loss')
    ]

    history = model.fit(train_generator, epochs=350, validation_data=val_generator, callbacks=callbacks)

    all_fold_histories.append(history.history)

    # Load the best model for this fold and evaluate
    model_path = f'{MODEL_DIR}/model_{fold+1}.h5'
    model = load_model(model_path)
    scores = model.evaluate(val_generator)
    val_accuracy = scores[1]  # Assuming [0] is loss and [1] is accuracy

    if val_accuracy > best_model_accuracy:
        best_model_accuracy = val_accuracy
        best_model_path = model_path

    # Predictions and metrics calculation
    val_preds = model.predict(val_generator)
    val_preds_labels = np.argmax(val_preds, axis=1)
    true_labels = val_generator.classes

    precision = precision_score(true_labels, val_preds_labels, average='macro')
    recall = recall_score(true_labels, val_preds_labels, average='macro')
    f1 = f1_score(true_labels, val_preds_labels, average='macro')

    fold_metrics.append((precision, recall, f1))

# Assuming best_model_path is set to the path of the best model
if best_model_path:
    best_model = load_model(best_model_path)
    best_model.save(f'{MODEL_DIR}/Spots_Distribution_best_model_overall.h5')
    print("Best model saved successfully.")

# Calculate average metrics for plotting
avg_history = {key: np.mean([fh[key] for fh in all_fold_histories], axis=0) for key in all_fold_histories[0]}

# Plotting
plt.figure(figsize=(12, 5))

# Plot Average Training and Validation Accuracy
plt.subplot(1, 2, 1)
plt.plot(avg_history['accuracy'], label='Avg Training Accuracy')
plt.plot(avg_history['val_accuracy'], label='Avg Validation Accuracy')
plt.title('Average Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Average Training and Validation Loss
plt.subplot(1, 2, 2)
plt.plot(avg_history['loss'], label='Avg Training Loss')
plt.plot(avg_history['val_loss'], label='Avg Validation Loss')
plt.title('Average Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# Assuming you want to plot metrics for the best model
# Plotting confusion matrix for the best model as an example
# You would need to redo predictions on your validation set or a test set with the best model
best_val_preds = best_model.predict(val_generator)
best_val_preds_labels = np.argmax(best_val_preds, axis=1)
cm = confusion_matrix(true_labels, best_val_preds_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.show()

# Plotting precision, recall, and F1 for each fold
precisions, recalls, f1s = zip(*fold_metrics)
plt.figure(figsize=(12, 5))

# Plotting Precision for each fold
plt.plot(precisions, label='Precision', marker='o', linestyle='-', color='blue')
plt.plot(recalls, label='Recall', marker='s', linestyle='-', color='red')
plt.plot(f1s, label='F1 Score', marker='^', linestyle='-', color='green')

plt.title('Precision, Recall, F1 Score for Each Fold')
plt.xlabel('Fold Number')
plt.ylabel('Score')
plt.xticks(range(len(precisions)), [f"Fold {i+1}" for i in range(len(precisions))])
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



## 8. Training variant C

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from tensorflow.keras.models import load_model, save_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator



def get_data_generator(image_paths, labels, batch_size=32, is_validation=False):
    datagen = ImageDataGenerator(rescale=1./255)
    shuffle = not is_validation
    return datagen.flow_from_dataframe(
        dataframe=pd.DataFrame({'filename': image_paths, 'class': labels.astype(str)}),
        x_col='filename',
        y_col='class',
        target_size=(224, 224),
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=shuffle)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
labels_np = np.array(labels)

best_model_accuracy = -1
best_model_path = ''
fold_metrics = []
all_fold_histories = []

for fold, (train_idx, val_idx) in enumerate(skf.split(images, labels_np)):
    print(f"Training on fold {fold+1}/{n_splits}...")
    X_train, X_val = np.array(images)[train_idx], np.array(images)[val_idx]
    y_train, y_val = np.array(labels)[train_idx], np.array(labels)[val_idx]

    model = create_model()

    train_generator = get_data_generator(X_train, y_train, is_validation=False)
    val_generator = get_data_generator(X_val, y_val, is_validation=True)

    callbacks = [
        ModelCheckpoint(f'{MODEL_DIR}/model_{fold+1}.h5', save_best_only=True, monitor='val_accuracy'),
        EarlyStopping(patience=10, monitor='val_loss'),
        ReduceLROnPlateau(patience=5, monitor='val_loss')
    ]

    history = model.fit(train_generator, epochs=350, validation_data=val_generator, callbacks=callbacks)

    all_fold_histories.append(history.history)

    model_path = f'{MODEL_DIR}/model_{fold+1}.h5'
    model = load_model(model_path)
    scores = model.evaluate(val_generator)
    val_accuracy = scores[1]

    if val_accuracy > best_model_accuracy:
        best_model_accuracy = val_accuracy
        best_model_path = model_path

    val_generator.reset()
    val_preds = model.predict(val_generator)
    val_preds_labels = np.argmax(val_preds, axis=1)
    true_labels = val_generator.classes

    precision = precision_score(true_labels, val_preds_labels, average='macro')
    recall = recall_score(true_labels, val_preds_labels, average='macro')
    f1 = f1_score(true_labels, val_preds_labels, average='macro')

    fold_metrics.append((precision, recall, f1))

# Load the best model and reconfigure the validation generator if necessary
if best_model_path:
    best_model = load_model(best_model_path)
    best_model.save(f'{MODEL_DIR}/Spots_Distribution_best_model_overall.h5')
    print("Best model saved successfully.")



## 9. Additional diagnostics (optional)

In [ ]:
# Calculate average metrics for plotting
avg_history = {key: np.mean([fh[key] for fh in all_fold_histories], axis=0) for key in all_fold_histories[0]}

# Plotting
plt.figure(figsize=(12, 5))

# Plot Average Training and Validation Accuracy
plt.subplot(1, 2, 1)
plt.plot(avg_history['accuracy'], label='Avg Training Accuracy')
plt.plot(avg_history['val_accuracy'], label='Avg Validation Accuracy')
plt.title('Average Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Plot Average Training and Validation Loss
plt.subplot(1, 2, 2)
plt.plot(avg_history['loss'], label='Avg Training Loss')
plt.plot(avg_history['val_loss'], label='Avg Validation Loss')
plt.title('Average Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

# Assume reconfiguration or resetting of val_generator for a comprehensive validation dataset if necessary
# This step is crucial for ensuring the confusion matrix is calculated on the correct dataset

# Example assuming a dedicated evaluation dataset setup
# val_generator = get_data_generator(evaluation_images, evaluation_labels, is_validation=True)


# Recalculate predictions for the best model evaluation
# Create a new generator for comprehensive validation data
comprehensive_val_generator = get_data_generator(comprehensive_val_images, comprehensive_val_labels, is_validation=True)

# Evaluate the best model on the comprehensive validation dataset
best_model.evaluate(comprehensive_val_generator)

# Reset the generator if needed (depends on the generator implementation)
comprehensive_val_generator.reset()

# Recalculate predictions for the best model evaluation
best_val_preds = best_model.predict(comprehensive_val_generator)
best_val_preds_labels = np.argmax(best_val_preds, axis=1)
true_labels = comprehensive_val_generator.classes  # Make sure this matches the dataset intended for final evaluation
cm = confusion_matrix(true_labels, best_val_preds_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted labels')
plt.ylabel('True labels')
plt.show()

# Continue with the rest of the plotting logic as previously defined
# Plotting precision, recall, and F1 for each fold
precisions, recalls, f1s = zip(*fold_metrics)
plt.figure(figsize=(12, 5))

# Plotting Precision for each fold
plt.plot(precisions, label='Precision', marker='o', linestyle='-', color='blue')
plt.plot(recalls, label='Recall', marker='s', linestyle='-', color='red')
plt.plot(f1s, label='F1 Score', marker='^', linestyle='-', color='green')

plt.title('Precision, Recall, F1 Score for Each Fold')
plt.xlabel('Fold Number')
plt.ylabel('Score')
plt.xticks(range(len(precisions)), [f"Fold {i+1}" for i in range(len(precisions))])
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()



In [ ]:
plt.figure(figsize=(24, 10))

# Training and Validation Accuracy
plt.subplot(1, 2, 1)
for fold_history in all_fold_histories:
    plt.plot(fold_history['accuracy'], label='Training Accuracy (Fold)')
    plt.plot(fold_history['val_accuracy'], label='Validation Accuracy (Fold)')
plt.title('Training and Validation Accuracy Across Folds')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='upper left')

# Training and Validation Loss
plt.subplot(1, 2, 2)
for fold_history in all_fold_histories:
    plt.plot(fold_history['loss'], label='Training Loss (Fold)')
    plt.plot(fold_history['val_loss'], label='Validation Loss (Fold)')
plt.title('Training and Validation Loss Across Folds')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()
